In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_parquet("/Users/arjunprakashrao/Drive/projects/Polymarket-lab/cricket_arbitrage/internal_data/generated/dls_dataset.parquet")
df.head()

,match_id,season,venue,city,date,innings,team,opponent,batter,bowler,...,is_dot,is_six,is_boundary,is_wicket,phase,censored,target_total,final_total,team_id,opponent_team_id
0,14503badd17c484,2024,Pallekele International Cricket Stadium,Kandy,2024-07-30,1,India,Sri Lanka,YBK Jaiswal,C Wickramasinghe,...,1,0,0,0,powerplay,False,NaN,137,40,89
1,14503badd17c484,2024,Pallekele International Cricket Stadium,Kandy,2024-07-30,1,India,Sri Lanka,YBK Jaiswal,C Wickramasinghe,...,0,0,0,0,powerplay,False,NaN,137,40,89
2,14503badd17c484,2024,Pallekele International Cricket Stadium,Kandy,2024-07-30,1,India,Sri Lanka,Shubman Gill,C Wickramasinghe,...,1,0,0,0,powerplay,False,NaN,137,40,89
3,14503badd17c484,2024,Pallekele International Cricket Stadium,Kandy,2024-07-30,1,India,Sri Lanka,Shubman Gill,C Wickramasinghe,...,0,0,0,0,powerplay,False,NaN,137,40,89
4,14503badd17c484,2024,Pallekele International Cricket Stadium,Kandy,2024-07-30,1,India,Sri Lanka,YBK Jaiswal,C Wickramasinghe,...,0,0,0,0,powerplay,False,NaN,137,40,89


In [3]:
rows = []

df = df.sort_values(
    ["match_id", "innings", "over_number", "ball_in_over"]
)

df["ball_index"] = (
    df.groupby(["match_id", "innings"])
      .cumcount()
)

for (match_id, innings), g in df.groupby(["match_id", "innings"]):

    g = g.reset_index(drop=True)

    is_second_innings = 1 if innings == 2 else 0

    # Ensure we don't go out of bounds for the last 6 balls of an inning
    for i in range(len(g) - 6):

        cur = g.loc[i]
        
        # Grab the next 6 balls (loc is inclusive on both ends, so i+1 to i+6 gets exactly 6 balls)
        nxt = g.loc[i+1:i+6]

        # Calculate aggregates over the next 6 balls
        runs_next_6_balls = nxt["total_runs_ball"].sum()
        wickets_next_6_balls = nxt["is_wicket"].sum()
        wicket_event = int(wickets_next_6_balls > 0)

        legal_balls = max(cur["legal_balls_bowled"], 1)

        rr = cur["current_score"] / legal_balls * 6

        overs_remaining = (120 - legal_balls) / 6

        target = cur["target_total"] if is_second_innings else 0

        if is_second_innings and target > 0:
            runs_remaining = max(target - cur["current_score"], 0)
            balls_remaining = max(120 - legal_balls, 1)
            rrr = runs_remaining / balls_remaining * 6
        else:
            rrr = 0

        rows.append({
            "match_id": match_id,
            "innings": innings,
            "is_second_innings": is_second_innings,
            "rr": rr,
            "required_run_rate": rrr,
            "overs_remaining": overs_remaining,
            "wickets_in_hand": cur["wickets_in_hand"],
            "current_score": cur["current_score"],
            "target_total": target,
            "target_runs_next_6_balls": runs_next_6_balls,
            "wickets_next_6_balls": wickets_next_6_balls,
            "wicket_event": wicket_event
        })

train_df = pd.DataFrame(rows)


In [4]:
train_df

,match_id,innings,is_second_innings,rr,required_run_rate,overs_remaining,wickets_in_hand,current_score,target_total,target_runs_next_6_balls,wickets_next_6_balls,wicket_event
0,002e868e1a09d75,1,0,0.000000,0.000000,19.833333,10,0,0.0,3,2,1
1,002e868e1a09d75,1,0,0.000000,0.000000,19.833333,10,0,0.0,3,2,1
2,002e868e1a09d75,1,0,0.000000,0.000000,19.666667,10,0,0.0,4,1,1
3,002e868e1a09d75,1,0,0.000000,0.000000,19.500000,9,0,0.0,2,2,1
4,002e868e1a09d75,1,0,3.000000,0.000000,19.333333,9,2,0.0,6,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...
680391,fffe68d09df897b,2,1,8.482759,1.272727,5.500000,7,123,130.0,5,1,1
680392,fffe68d09df897b,2,1,8.454545,1.125000,5.333333,7,124,130.0,5,1,1
680393,fffe68d09df897b,2,1,8.359551,1.161290,5.166667,7,124,130.0,3,1,1
680394,fffe68d09df897b,2,1,8.494382,0.774194,5.166667,7,126,130.0,2,2,1


In [5]:
features = [
    "rr",
    "required_run_rate",
    "wickets_in_hand",
    "is_second_innings",
    "current_score",
    "overs_remaining"
]


In [6]:
import statsmodels.api as sm


X = train_df[features]
X = sm.add_constant(X)

y_runs = train_df["target_runs_next_6_balls"]

model_runs = sm.NegativeBinomial(y_runs, X).fit()
print(model_runs.summary())

Optimization terminated successfully.
         Current function value: 2.750528
         Iterations: 24
         Function evaluations: 33
         Gradient evaluations: 33
                        NegativeBinomial Regression Results                         
Dep. Variable:     target_runs_next_6_balls   No. Observations:               680396
Model:                     NegativeBinomial   Df Residuals:                   680389
Method:                                 MLE   Df Model:                            6
Date:                      Sun, 08 Mar 2026   Pseudo R-squ.:                 0.01531
Time:                              12:01:27   Log-Likelihood:            -1.8714e+06
converged:                             True   LL-Null:                   -1.9005e+06
Covariance Type:                  nonrobust   LLR p-value:                     0.000
                        coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------

In [7]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.api import Logit
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score

X = train_df[features]
y = train_df["wicket_event"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train = sm.add_constant(X_train)
X_test = sm.add_constant(X_test)

wicket_model = Logit(y_train, X_train).fit()

y_pred_prob = wicket_model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print(wicket_model.summary())

Optimization terminated successfully.
         Current function value: 0.595275
         Iterations 5
Precision: 0.4116
Recall: 0.0029
                           Logit Regression Results                           
Dep. Variable:           wicket_event   No. Observations:               544316
Model:                          Logit   Df Residuals:                   544309
Method:                           MLE   Df Model:                            6
Date:                Sun, 08 Mar 2026   Pseudo R-squ.:                 0.01158
Time:                        12:01:28   Log-Likelihood:            -3.2402e+05
converged:                       True   LL-Null:                   -3.2781e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                -0.4237      0.025    -17.050      0.000

In [8]:
import json

run_coeffs = model_runs.params.to_dict()

with open("run_model_coeffs.json", "w") as f:
    json.dump(run_coeffs, f)

wicket_coeffs = wicket_model.params.to_dict()

with open("wicket_model_coeffs.json", "w") as f:
    json.dump(wicket_coeffs, f)

In [14]:
with open("/Users/arjunprakashrao/Drive/projects/Polymarket-lab/cricket_arbitrage/bots/run_model_coeffs.json", "w") as f:
    json.dump(run_coeffs, f)

with open("/Users/arjunprakashrao/Drive/projects/Polymarket-lab/cricket_arbitrage/bots/wicket_model_coeffs.json", "w") as f:
    json.dump(wicket_coeffs, f)

In [9]:
import json

with open("run_model_coeffs.json") as f:
    RUN = json.load(f)

with open("wicket_model_coeffs.json") as f:
    WICKET = json.load(f)

In [10]:
import numpy as np
from scipy.stats import nbinom
import math

def predict_mu(rr, rrr, overs_remaining, wih, is_second, score):

    z = (
        RUN["const"]
        + RUN["rr"] * rr
        + RUN["required_run_rate"] * rrr
        + RUN["overs_remaining"] * overs_remaining
        + RUN["wickets_in_hand"] * wih
        + RUN["is_second_innings"] * is_second
        + RUN["current_score"] * score
    )

    return np.exp(z)

def sample_runs(mu):

    alpha = RUN["alpha"]
    r = 1 / alpha
    p = r / (r + mu)

    return np.random.negative_binomial(r, p)

def predict_p_wicket(rr, rrr, overs_remaining, wih, is_second, score):

    z = (
        WICKET["const"]
        + WICKET["rr"] * rr
        + WICKET["required_run_rate"] * rrr
        + WICKET["overs_remaining"] * overs_remaining
        + WICKET["wickets_in_hand"] * wih
        + WICKET["is_second_innings"] * is_second
        + WICKET["current_score"] * score
    )

    return 1 / (1 + np.exp(-z))

# Monte Carlo

In [11]:
MAX_BALLS = 120
MAX_WICKETS = 10
STEP_BALLS = 6

In [12]:
initial = init_state(
    score=60,
    wickets=3,
    balls=60,
    target=150
)

p_win = monte_carlo_win_prob(initial, N=100000)
print("Win Probability:", round(p_win, 3))

NameError: name 'init_state' is not defined